# Social Engagement Trend Analysis

## Project Overview
This notebook analyzes social media post performance over time, comparing impressions, engagement rate, post types, hashtag usage, and timing patterns. We use realistic synthetic data modeled on typical brand social media analytics.

## Learning Objectives
- Compute and compare engagement rate metrics across post types and platforms.
- Identify optimal posting times using day-of-week and hour analysis.
- Analyze hashtag effectiveness on reach and engagement.
- Detect engagement trends over time and seasonal patterns.

## Problem Statement
A brand's social media team publishes content across platforms and formats. They need to understand which content types drive the most engagement, what posting schedule works best, and how engagement trends are evolving.

## Why This Project Matters
Social media engagement directly affects brand visibility, customer acquisition, and community growth. Data-driven content strategy outperforms intuition-based posting by identifying what actually resonates with the audience.

## Dataset Overview
Synthetic dataset of ~3,000 social media posts over 12 months with fields: date, platform, post type, impressions, likes, comments, shares, hashtag count, and posting hour.

## Dataset Source and License Notes
Synthetic data generated in-notebook. No external dependencies.

## Environment Setup

In [ ]:
import importlib, subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## Imports

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')
np.random.seed(42)

## Configuration / Constants

In [ ]:
PLATFORMS = ['Instagram', 'Twitter/X', 'LinkedIn', 'Facebook']
POST_TYPES = ['Image', 'Video', 'Carousel', 'Text', 'Story', 'Reel']
N_POSTS = 3000

## Dataset Generation

In [ ]:
rng = np.random.default_rng(42)

platform_reach = {'Instagram': 5000, 'Twitter/X': 3000, 'LinkedIn': 2000, 'Facebook': 4000}
type_eng_mult = {'Image': 1.0, 'Video': 1.4, 'Carousel': 1.3, 'Text': 0.6, 'Story': 0.9, 'Reel': 1.6}

records = []
for _ in range(N_POSTS):
    platform = rng.choice(PLATFORMS, p=[0.35, 0.25, 0.15, 0.25])
    post_type = rng.choice(POST_TYPES, p=[0.25, 0.20, 0.15, 0.15, 0.10, 0.15])
    date = pd.Timestamp('2024-07-01') + pd.Timedelta(days=int(rng.integers(0, 365)))
    hour = int(rng.choice([8,9,10,11,12,13,14,17,18,19,20,21], p=[0.05,0.08,0.10,0.10,0.12,0.08,0.07,0.08,0.10,0.08,0.08,0.06]))
    hashtags = int(rng.integers(0, 12))

    base_impressions = platform_reach[platform] * rng.lognormal(0, 0.5)
    hashtag_boost = 1 + hashtags * 0.03
    impressions = int(max(100, base_impressions * hashtag_boost))

    base_eng = 0.035 * type_eng_mult[post_type]
    # Time-of-day effect
    time_mult = 1.2 if 10 <= hour <= 13 else (1.0 if 17 <= hour <= 20 else 0.85)
    eng_rate = base_eng * time_mult * rng.lognormal(0, 0.3)

    likes = int(impressions * eng_rate * rng.uniform(0.5, 1.0))
    comments = int(likes * rng.uniform(0.05, 0.25))
    shares = int(likes * rng.uniform(0.02, 0.15))

    records.append({
        'Date': date, 'Platform': platform, 'Post Type': post_type,
        'Hour': hour, 'Hashtags': hashtags,
        'Impressions': impressions, 'Likes': likes,
        'Comments': comments, 'Shares': shares,
    })

df = pd.DataFrame(records)
df['Engagements'] = df['Likes'] + df['Comments'] + df['Shares']
df['Engagement Rate %'] = np.where(df['Impressions'] > 0, df['Engagements'] / df['Impressions'] * 100, 0)
df['Day of Week'] = df['Date'].dt.day_name()

print(f'Dataset shape: {df.shape}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Rows: {len(df)}, Columns: {len(df.columns)}')
print(f'Nulls:\n{df.isna().sum().to_string()}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Platforms: {sorted(df["Platform"].unique())}')
print(f'Post types: {sorted(df["Post Type"].unique())}')

## Exploratory Data Analysis

### Overall Metrics

In [ ]:
print('Overall Metrics:')
print(f'  Total posts: {len(df):,}')
print(f'  Total impressions: {df["Impressions"].sum():,.0f}')
print(f'  Total engagements: {df["Engagements"].sum():,.0f}')
print(f'  Avg engagement rate: {df["Engagement Rate %"].mean():.2f}%')
print(f'  Avg impressions/post: {df["Impressions"].mean():,.0f}')

### Engagement by Platform

In [ ]:
plat_perf = df.groupby('Platform').agg(
    Posts=('Impressions', 'count'),
    Avg_Impressions=('Impressions', 'mean'),
    Avg_Engagement_Rate=('Engagement Rate %', 'mean'),
    Total_Engagements=('Engagements', 'sum'),
).sort_values('Avg_Engagement_Rate', ascending=False).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(plat_perf.index, plat_perf['Avg_Engagement_Rate'], color='steelblue', edgecolor='black')
axes[0].set_title('Avg Engagement Rate by Platform')
axes[0].set_ylabel('Engagement Rate %')

axes[1].bar(plat_perf.index, plat_perf['Avg_Impressions'], color='teal', edgecolor='black')
axes[1].set_title('Avg Impressions by Platform')
axes[1].set_ylabel('Impressions')

plt.tight_layout()
plt.show()
print(plat_perf.to_string())

### Engagement by Post Type

In [ ]:
type_perf = df.groupby('Post Type').agg(
    Posts=('Impressions', 'count'),
    Avg_Engagement_Rate=('Engagement Rate %', 'mean'),
    Avg_Impressions=('Impressions', 'mean'),
).sort_values('Avg_Engagement_Rate', ascending=False).round(2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(type_perf.index, type_perf['Avg_Engagement_Rate'], color='seagreen', edgecolor='black')
ax.set_xlabel('Avg Engagement Rate %')
ax.set_title('Engagement Rate by Post Type')
plt.tight_layout()
plt.show()
print(type_perf.to_string())

### Optimal Posting Time — Hour of Day

In [ ]:
hour_perf = df.groupby('Hour').agg(
    Posts=('Impressions', 'count'),
    Avg_ER=('Engagement Rate %', 'mean'),
    Avg_Impressions=('Impressions', 'mean'),
).round(2)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(hour_perf.index, hour_perf['Avg_ER'], color='darkorange', edgecolor='black', alpha=0.7)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Engagement Rate %')
ax.set_title('Engagement Rate by Posting Hour')
ax.set_xticks(hour_perf.index)
plt.tight_layout()
plt.show()

### Day of Week Analysis

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_perf = df.groupby('Day of Week').agg(
    Posts=('Impressions', 'count'),
    Avg_ER=('Engagement Rate %', 'mean'),
).reindex(dow_order).round(2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(dow_perf.index, dow_perf['Avg_ER'], color='mediumpurple', edgecolor='black')
ax.set_ylabel('Avg Engagement Rate %')
ax.set_title('Engagement Rate by Day of Week')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### Hashtag Count vs Engagement

In [ ]:
hash_bins = pd.cut(df['Hashtags'], bins=[-1, 0, 3, 6, 11], labels=['0', '1-3', '4-6', '7-11'])
hash_perf = df.groupby(hash_bins, observed=True).agg(
    Posts=('Impressions', 'count'),
    Avg_ER=('Engagement Rate %', 'mean'),
    Avg_Impressions=('Impressions', 'mean'),
).round(2)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(hash_perf.index.astype(str), hash_perf['Avg_ER'], color='teal', edgecolor='black')
ax.set_xlabel('Hashtag Count')
ax.set_ylabel('Avg Engagement Rate %')
ax.set_title('Engagement Rate by Hashtag Usage')
plt.tight_layout()
plt.show()
print(hash_perf.to_string())

### Monthly Engagement Trend

In [ ]:
df['YearMonth'] = df['Date'].dt.to_period('M')
monthly = df.groupby('YearMonth').agg(
    Posts=('Impressions', 'count'),
    Avg_ER=('Engagement Rate %', 'mean'),
    Avg_Impressions=('Impressions', 'mean'),
    Total_Engagements=('Engagements', 'sum'),
).reset_index()
monthly['date'] = monthly['YearMonth'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['date'], monthly['Avg_ER'], 'o-', color='seagreen', linewidth=2)
ax.set_title('Monthly Avg Engagement Rate Trend')
ax.set_ylabel('Engagement Rate %')
plt.tight_layout()
plt.show()

### Platform × Post Type Heatmap

In [ ]:
cross = df.pivot_table(values='Engagement Rate %', index='Post Type', columns='Platform', aggfunc='mean')
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(cross, annot=True, fmt='.2f', cmap='YlGn', ax=ax, linewidths=0.5)
ax.set_title('Avg Engagement Rate (%) — Post Type × Platform')
plt.tight_layout()
plt.show()

## Executive Summary

In [ ]:
best_type = type_perf['Avg_Engagement_Rate'].idxmax()
best_platform = plat_perf['Avg_Engagement_Rate'].idxmax()
best_hour = hour_perf['Avg_ER'].idxmax()

insights = [
    f'Total posts analyzed: {len(df):,}',
    f'Overall avg engagement rate: {df["Engagement Rate %"].mean():.2f}%',
    f'Best post type: {best_type} ({type_perf.loc[best_type, "Avg_Engagement_Rate"]:.2f}% ER)',
    f'Best platform: {best_platform} ({plat_perf.loc[best_platform, "Avg_Engagement_Rate"]:.2f}% ER)',
    f'Best posting hour: {best_hour}:00 ({hour_perf.loc[best_hour, "Avg_ER"]:.2f}% ER)',
    'Reels and Videos consistently outperform Text and Image posts',
    'Moderate hashtag usage (4-6) tends to boost reach without hurting engagement rate',
]
for item in insights:
    print(f'  * {item}')

## Limitations
- Synthetic data — real social data includes algorithm changes, ad boosts, and audience growth dynamics.
- No follower count or paid promotion data to normalize engagement.
- Hashtag analysis is simplified — real analysis would need individual hashtag tracking.

## Recommendations
- Prioritize Reels and Video content for higher engagement rates.
- Post between 10 AM and 1 PM for optimal engagement.
- Use 4-6 hashtags per post as a practical sweet spot.
- Track monthly trends to detect algorithm or audience shifts early.

## Common Mistakes
- Comparing raw likes across platforms with different audience sizes — always use rate metrics.
- Ignoring post volume when evaluating — a format with 5 posts may show high variance.
- Not controlling for paid boost when analyzing organic engagement.

## Mini Challenge
1. Find the single best-performing post type × platform × hour combination.
2. Build a "virality index" (shares / impressions) and rank post types by it.
3. Compare weekend vs weekday engagement — does it differ by platform?
4. Detect if any month shows a statistically significant drop in engagement rate.

## Final Summary / Key Takeaways
Social engagement analysis requires consistent rate metrics, time-based trend analysis, and content type comparison. The best-performing content combines the right format (video/reel), timing (midday), and platform fit. Always segment results before averaging to avoid Simpson's paradox.